# Feature centering for fast few-shot video adaptation

Run setup, data preparation, feature caching, and episode generation below. Mount Drive and choose persistent storage before caching. A GPU is recommended for the initial frozen encoder cache. The final paired heads use CPU.

The current method subtracts a class-balanced mean fitted only on training support features. Both methods execute exactly **50 optimizer updates** with the first 50 updates of a common 300-update learning-rate schedule. The available training episode file contains 300 episodes; only the first 50 are consumed.

Historical small-set AP: fixed 51.43 ± 0.93, centered 52.04 ± 0.08 across three training seeds. It contains 3 new classes, 6 videos, and 18 distinct frames. A later frozen-model check on 6 further unseen classes, 24 videos, and 192 frames gave fixed/centered AP 38.61/35.14. All three centered runs were worse. The larger check did not confirm faster adaptation. See `results/larger_confirmation/README.md` and `results/revision_v5/README.md` for both results. This notebook reproduces the historical limited-update study. Earlier failed experiments remain available.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

os.environ["USE_TF"] = "0"
BASE = (Path("/content") if Path("/content").is_dir() else
        Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd())
REPO = BASE / "adaptive-fsvod"
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "https://github.com/hskad/adaptive-fsvod.git", str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
# Editable installation affects new Python processes. Also expose src to the
# notebook kernel, which was already running when pip created its .pth file.
PACKAGE_FILE = Path(subprocess.check_output(
    [sys.executable, "-c", "import adaptive_fsvod; print(adaptive_fsvod.__file__)"],
    text=True).strip()).resolve()
SOURCE_PATH = str(PACKAGE_FILE.parent.parent)
if SOURCE_PATH not in sys.path:
    sys.path.insert(0, SOURCE_PATH)
import importlib
importlib.invalidate_caches()
import adaptive_fsvod
print("Project imports:", adaptive_fsvod.__file__)
subprocess.run(["git", "rev-parse", "HEAD"], check=True)


In [ ]:
import torch
assert torch.cuda.is_available(), "Select a GPU runtime before feature extraction."
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)


## Set paths

For Colab persistence, mount Drive in a separate cell with `from google.colab import drive; drive.mount('/content/drive')`. Then set `EXPERIMENT_DIR`, for example, to `/content/drive/MyDrive/adaptive-fsvod-experiment`. Do not put experiment outputs in a read-only Kaggle input directory.


In [ ]:
EXPERIMENT_DIR = BASE / "adaptive-fsvod-experiment"  # Change to persistent storage.
IMAGE_ROOT = EXPERIMENT_DIR / "images"             # Selected frames download here.
SEED = 42
STEPS = 300
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

def run(module, **flags):
    command = [sys.executable, "-m", "adaptive_fsvod." + module]
    for name, value in flags.items():
        if isinstance(value, bool):
            if value:
                command.append("--" + name.replace("_", "-"))
            continue
        command.append("--" + name.replace("_", "-"))
        if isinstance(value, (list, tuple)):
            command.extend(map(str, value))
        else:
            command.append(str(value))
    subprocess.run(command, check=True)


In [ ]:
RAW = EXPERIMENT_DIR / "raw"
RAW.mkdir(exist_ok=True)
ARCHIVE = RAW / "fsytv-annotations.zip"
BOXES = RAW / "fsytv-boxes.json"
if not ARCHIVE.exists():
    subprocess.run([sys.executable, "-m", "gdown", "1-m5Qcuxj4h8rryqnELtgnvOtTtWJ_Vys", "-O", str(ARCHIVE)], check=True)
if not BOXES.exists():
    run("convert_fsytv", archive=ARCHIVE, output=BOXES)

DATA = EXPERIMENT_DIR / "pilot"
run("prepare", annotation=BOXES, images_root=IMAGE_ROOT, output=DATA,
    class_counts=[6, 3, 3], max_videos_per_class=3, frame_stride=4, seed=SEED, skip_image_check=True)
run("download_frames", manifests=[DATA / f"{split}.json" for split in ["train", "val", "test"]],
    images_root=IMAGE_ROOT, range_cache=EXPERIMENT_DIR / "download_ranges")
from adaptive_fsvod.data import VideoManifest
for split in ["train", "val", "test"]:
    print(split, VideoManifest(DATA / f"{split}.json").validate_images())


## Cache frozen features

All methods share these caches. The encoder runs at its native resolution. Only the top 128 tokens are retained afterward. The cache can resume an interrupted extraction. Settings and annotation checksums must match.


In [ ]:
from adaptive_fsvod.cache import FrozenEncoder, build_cache
from adaptive_fsvod.data import VideoManifest

CACHE = EXPERIMENT_DIR / "cache"
encoder = FrozenEncoder(device="cuda", revision="cfd3195ba4ea9592eec887ded089f4c08eff231d", candidates=128)
for split in ["train", "val", "test"]:
    print(split, build_cache(VideoManifest(DATA / f"{split}.json"), CACHE / split, encoder, per_class=16))
del encoder
torch.cuda.empty_cache()


In [ ]:
for split, count, episode_seed in [("train", 300, 42), ("val", 20, 43), ("test", 30, 44)]:
    flags = dict(manifest=DATA / f"{split}.json", cache=CACHE / split,
                 output=DATA / f"{split}_episodes.json", episodes=count,
                 n_way=2, k_shot=1, seed=episode_seed)
    if split == "train":
        flags["max_frames"] = 4
    run("episodes", **flags)


## Run the current experiment

The cell below resumes existing checkpoints, runs an equal-budget validation search and three-seed comparison, prepares the fixed fresh subset, caches it, freezes input hashes, evaluates, verifies all metrics and inference weights, and exports a CSV, figure, and checkpoint ZIP. Confirmation encoder extraction uses CPU to match the recorded result; it can take several minutes. No manual copying between stages is needed. Errors stop execution.


In [ ]:
def run_script(script, *arguments):
    command = [sys.executable, "-u", str(REPO / "scripts" / script), *map(str, arguments)]
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        status = process.wait()
        if status:
            raise subprocess.CalledProcessError(status, command)

run_script("run_fast_study.py", "--experiment-dir", EXPERIMENT_DIR,
           "--stage", "all", "--threads", 4)
run_script("export_fast_results.py", "--experiment-dir", EXPERIMENT_DIR,
           "--output", EXPERIMENT_DIR / "exports" / "verified_fast_results")


In [ ]:
from IPython.display import display
from PIL import Image
RESULTS = EXPERIMENT_DIR / "exports" / "verified_fast_results"
print((RESULTS / "comparison.csv").read_text())
display(Image.open(RESULTS / "fast_adaptation.png"))
print("Inference weights:", EXPERIMENT_DIR / "exports" / "fast_adaptation_inference.zip")
